In [ ]:
from google.colab import files
import pandas as pd
import os

print("=" * 60)
print("UPLOAD YOUR BRAVE ISSUES FILE")
print("Upload: df_brave_issues_02080308_csv.gzip")
print("=" * 60)

uploaded = files.upload()

# Find the uploaded file
uploaded_filename = list(uploaded.keys())[0]
print(f"\nUploaded: {uploaded_filename}")

try:
    df = pd.read_csv(uploaded_filename, compression='gzip')
    print("Loaded with gzip compression")
except:
    try:
        df = pd.read_csv(uploaded_filename)
        print("Loaded as plain CSV")
    except:
        import io
        df = pd.read_csv(io.BytesIO(uploaded[uploaded_filename]))
        print("Loaded from bytes")

print(f"Loaded {len(df)} rows")

UPLOAD YOUR BRAVE ISSUES FILE
Upload: df_brave_issues_02080308_csv.gzip


Saving df_brave_issues_02080308.csv.gzip to df_brave_issues_02080308.csv (1).gzip

Uploaded: df_brave_issues_02080308.csv (1).gzip
Loaded as plain CSV
Loaded 556 rows


In [ ]:
!pip install -q transformers bitsandbytes accelerate pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted!


In [ ]:
import re

# Filter: actual issues (not PRs), with meaningful body text
issues = df[
    df['pull_request'].isna() &
    df['body'].notna() &
    (df['body'].str.len() > 50)
].copy()

# Extract useful metadata
issues['has_bug_label'] = issues['labels'].str.contains('bug', case=False, na=False)
issues['is_closed'] = issues['state'] == 'closed'

def clean_issue_body(body):
    """Remove GitHub issue template boilerplate, keep actual content."""
    if pd.isna(body):
        return ""
    body = str(body)

    # Remove HTML comments
    body = re.sub(r'<!--.*?-->', '', body, flags=re.DOTALL)

    # Remove common template headers
    template_markers = [
        "Have you searched for similar issues?",
        "PLEASE USE THE TEMPLATE BELOW",
        "INSUFFICIENT INFO WILL GET THE ISSUE CLOSED",
        "Submitter Checklist:",
        "Add brave-browser issue bellow",
    ]
    for marker in template_markers:
        body = body.replace(marker, '')

    # Remove image links
    body = re.sub(r'<img[^>]*alt="([^"]*)"[^>]*>', r'\1', body)
    body = re.sub(r'<img[^>]*>', '', body)

    # Remove URLs but keep context
    body = re.sub(r'https?://\S+', '[URL]', body)

    # Clean up whitespace
    body = re.sub(r'\n{3,}', '\n\n', body)
    body = body.strip()

    return body

issues['clean_body'] = issues['body'].apply(clean_issue_body)

def build_issue_context(row):
    """Create a concise summary of the issue for the prompt."""
    parts = []
    parts.append(f"Title: {row['title']}")

    if row['clean_body']:
        body_excerpt = row['clean_body'][:800]
        parts.append(f"Description: {body_excerpt}")

    parts.append(f"Status: {'Closed/Fixed' if row['is_closed'] else 'Open/Unfixed'}")
    parts.append(f"Type: {'Bug Report' if row['has_bug_label'] else 'Issue/Feature'}")

    return "\n".join(parts)

issues['issue_context'] = issues.apply(build_issue_context, axis=1)

print(f"\nUsable issues: {len(issues)}")
print(f"  Bug-labeled: {issues['has_bug_label'].sum()}")
print(f"  Open: {(~issues['is_closed']).sum()}")
print(f"  Closed: {issues['is_closed'].sum()}")

print("\n" + "=" * 60)
print("SAMPLE ISSUE CONTEXT:")
print("=" * 60)
print(issues['issue_context'].iloc[0])


Usable issues: 523
  Bug-labeled: 117
  Open: 276
  Closed: 247

SAMPLE ISSUE CONTEXT:
Title: Button text at the bottom of custom network form is cut off 
Description: ## Description 
Button text at the bottom of add custom network form is cut off/misaligned

## Steps to reproduce 

   1. Tap menu > settings
   2. Select Brave Wallet
   3. Create wallets, skip back up
   4. From the wallet dashboard, access the settings menu
   5. Select Networks 
   6. Select + Add Network
   7. Observe the button of the add network form

## Actual result 
'Add' button text is slightly cut off

![Screenshot_20220208-054629_Brave - Beta]([URL]


## Expected result
Text should not be cut off

## Issue reproduces how often 
Easily

## Version/Channel Information:


- Can you reproduce this issue with the current Play Store version? 
- Can you reproduce this issue with the current Play Store Beta version? 
- Can you reproduce this issue with th
Status: Closed/Fixed
Type: Bug Report


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"Model loaded! GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")


Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading model in 4-bit quantization...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded! GPU memory used: 4.1 GB


In [ ]:
PERSONA_TEMPLATES = [
    {
        "persona": "angry_short",
        "instruction": "Write a very short (1-2 sentences), angry 1-star Google Play Store review from a frustrated user who experienced this bug. Use informal language, maybe some caps. No technical jargon."
    },
    {
        "persona": "confused_user",
        "instruction": "Write a 2-3 sentence Google Play Store review from a confused, non-technical user who doesn't understand what went wrong. They just know something isn't working. Use simple vocabulary."
    },
    {
        "persona": "calm_reporter",
        "instruction": "Write a 2-3 sentence Google Play Store review from a calm user who matter-of-factly describes the problem. No anger, just stating what happened. Moderate detail."
    },
    {
        "persona": "sarcastic",
        "instruction": "Write a short, sarcastic Google Play Store review (1-2 sentences) from a user mocking the app for this issue. Dry humor."
    },
    {
        "persona": "desperate_user",
        "instruction": "Write a 2-3 sentence Google Play Store review from a desperate user who really needs the app to work but keeps hitting this bug. They are pleading for a fix."
    },
    {
        "persona": "technical_user",
        "instruction": "Write a 2-3 sentence Google Play Store review from a slightly technical user who describes the problem with some specificity but still uses everyday language, not developer jargon."
    },
    {
        "persona": "one_liner",
        "instruction": "Write a single short sentence Google Play Store review (under 15 words) that captures the essence of this bug from an end user perspective."
    },
    {
        "persona": "comparison_user",
        "instruction": "Write a 2-3 sentence Google Play Store review where the user compares this app unfavorably to a competitor because of this bug. Mention switching to another browser."
    },
    {
        "persona": "broken_english",
        "instruction": "Write a 1-2 sentence Google Play Store review from a non-native English speaker with slightly broken grammar describing this bug. Keep it natural and respectful."
    },
    {
        "persona": "detailed_complaint",
        "instruction": "Write a 3-4 sentence detailed Google Play Store review describing exactly what happens when the user encounters this bug, what they expected instead, and that they want it fixed in the next update."
    },
]

def build_prompt(issue_context, persona_template):
    """Build the generation prompt for Mistral Instruct format."""
    system_msg = (
        "You are simulating real Google Play Store user reviews for a mobile browser app called Brave. "
        "Write ONLY the review text. No labels, no ratings, no explanations, no quotes around it. "
        "The review must sound like a real user wrote it, not a developer and not an AI. "
        "Keep it natural and authentic. Do NOT mention bug numbers, issue IDs, or GitHub."
    )

    user_msg = (
        f"Here is a real bug report from the app's issue tracker:\n\n"
        f"{issue_context}\n\n"
        f"{persona_template['instruction']}\n\n"
        f"Write ONLY the review text, nothing else:"
    )

    # Mistral Instruct format
    prompt = f"[INST] {system_msg}\n\n{user_msg} [/INST]"
    return prompt


In [ ]:
import os
os.remove("checkpoint_brave.json")
print("Checkpoint deleted. Run Cell 6 again.")

Checkpoint deleted. Run Cell 6 again.


In [ ]:
def generate_review(prompt, max_new_tokens=150, temperature=0.9, top_p=0.95):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    review = tokenizer.decode(generated, skip_special_tokens=True).strip()
    review = review.strip('"').strip("'").strip()

    for prefix in ["Review:", "review:", "Rating:", "Here is", "Here's"]:
        if review.lower().startswith(prefix.lower()):
            review = review[len(prefix):].strip()
            review = review.lstrip(':').strip()

    return review

In [ ]:
import json, time, gc
from tqdm import tqdm
from google.colab import files

all_results = []
completed_issue_ids = set()

# Try resuming from local checkpoint
try:
    with open("checkpoint_brave.json", 'r') as f:
        checkpoint = json.load(f)
    all_results = checkpoint.get('results', [])
    completed_issue_ids = set(checkpoint.get('completed_ids', []))
    print(f"RESUMED: {len(completed_issue_ids)} issues, {len(all_results)} reviews")
except:
    print("Starting fresh")

remaining = issues[~issues['id'].isin(completed_issue_ids)]
print(f"Remaining: {len(remaining)} issues")

for idx, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining), desc="Generating")):
    issue_id = int(row['id'])
    for persona in PERSONA_TEMPLATES:
        prompt = build_prompt(row['issue_context'], persona)
        try:
            review = generate_review(prompt)
            if len(review) >= 5:
                all_results.append({
                    'issue_id': issue_id,
                    'issue_number': int(row['number']),
                    'issue_title': row['title'],
                    'issue_state': row['state'],
                    'has_bug_label': bool(row['has_bug_label']),
                    'persona': persona['persona'],
                    'synthetic_review': review,
                })
        except:
            continue

    completed_issue_ids.add(issue_id)

    if (idx + 1) % 10 == 0:
        # Save locally
        with open("checkpoint_brave.json", 'w') as f:
            json.dump({'results': all_results, 'completed_ids': list(completed_issue_ids)}, f)
        pd.DataFrame(all_results).to_csv("synthetic_reviews_brave.csv", index=False)

        # Auto-download to your computer
        files.download("synthetic_reviews_brave.csv")
        print(f"\n  Downloaded: {len(all_results)} reviews")

        gc.collect()
        torch.cuda.empty_cache()

# Final download
pd.DataFrame(all_results).to_csv("synthetic_reviews_brave.csv", index=False)
files.download("synthetic_reviews_brave.csv")
print(f"DONE: {len(all_results)} reviews")


Starting fresh
Remaining: 523 issues


Generating:   2%|▏         | 9/523 [08:36<8:14:51, 57.77s/it]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Generating:   2%|▏         | 10/523 [09:36<8:20:39, 58.56s/it]


  Downloaded: 100 reviews


... (17 outputs left out)

<IPython.core.display.Javascript object>

Generating:  10%|▉         | 50/523 [43:45<6:42:44, 51.09s/it]


  Downloaded: 500 reviews


Generating:  10%|█         | 53/523 [46:26<7:04:46, 54.23s/it]